In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/zalo-dataset/corpus.csv
/kaggle/input/zalo-dataset/crawled_test_75k.jsonl
/kaggle/input/zalo-dataset/crawled_test_5k.jsonl
/kaggle/input/qa-retrieval-rag-chunking/__results__.html
/kaggle/input/qa-retrieval-rag-chunking/index.faiss
/kaggle/input/qa-retrieval-rag-chunking/__huggingface_repos__.json
/kaggle/input/qa-retrieval-rag-chunking/__notebook__.ipynb
/kaggle/input/qa-retrieval-rag-chunking/__output__.json
/kaggle/input/qa-retrieval-rag-chunking/custom.css


In [ ]:
!pip install sentence-transformers
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 41.7 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
import numpy as np
import faiss
import json
import gc
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm.auto import tqdm

def calculate_metrics(retrieved_ids, ground_truth_ids, k):
    top_k_preds = set(retrieved_ids[:k])
    correct_preds = top_k_preds.intersection(ground_truth_ids)
    hit_val = 1 if len(correct_preds) > 0 else 0
    if len(ground_truth_ids) > 0:
        recall_val = len(correct_preds) / len(ground_truth_ids)
    else:
        recall_val = 0.0
    return hit_val, recall_val

def evaluate_retrieval_rag_final_optimized(
    test_data_path, 
    corpus_data_path, 
    faiss_index_path, 
    cross_encoder_name='BAAI/bge-reranker-v2-m3',
    k_values=[1, 5, 10, 30],
    top_k_rerank=30,
    top_k_retrieve=100,
    batch_process_size=100 
    ):
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    print("\n--- Loading Models ---")
    bi_encoder = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder', device=device)
    if device == 'cuda': bi_encoder.half() 
    
    print(f"Loading Cross-Encoder: {cross_encoder_name}")
    cross_encoder = CrossEncoder(cross_encoder_name, max_length=512, device=device)
    if device == 'cuda':
        cross_encoder.model.half()

    print(f"\n--- Loading Corpus Data ---")
    df_chunks = pd.read_csv(corpus_data_path, encoding='utf-8')
    
    all_chunk_ids = df_chunks['id'].astype(str).values
    
    print("Pre-processing corpus text...")
    if 'title' in df_chunks.columns and 'text' in df_chunks.columns:
        titles = df_chunks['title'].astype(str).values
        texts = df_chunks['text'].astype(str).values
        all_chunk_texts = ["Title: " + t + ". Content: " + txt for t, txt in zip(titles, texts)]
    else:
        all_chunk_texts = df_chunks['text'].astype(str).tolist()

    print(f"Loading FAISS index...")
    index = faiss.read_index(faiss_index_path)

    print(f"\n--- Loading Test Data ---")
    queries = []
    ground_truths = []
    
    with open(test_data_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line)
            q = item.get("question", "").strip()
            rels = item.get("relevant_articles", [])
            
            if q:
                queries.append(q)
                gt_ids = set()
                for article in rels:
                    if "ans_id" in article:
                        for a_id in article["ans_id"]:
                            gt_ids.add(str(a_id))
                ground_truths.append(gt_ids)
        
    total_queries = len(queries)
    print(f"Total Queries: {total_queries}")

    print(f"\n--- Stage 1: Retrieval (Bi-Encoder) ---")
    query_embeddings = bi_encoder.encode(queries, convert_to_numpy=True, batch_size=512, show_progress_bar=True)
    query_embeddings = query_embeddings.astype('float32')
    faiss.normalize_L2(query_embeddings)
    
    print(f"Searching Top {top_k_retrieve} candidates...")
    D, I = index.search(query_embeddings, top_k_retrieve)

    print(f"\n--- Stage 2: Reranking & Eval (Processing {batch_process_size} queries at a time) ---")
    
    metrics_retriever = {'hit': {k: 0 for k in k_values}, 'recall': {k: 0.0 for k in k_values}}
    metrics_reranker = {'hit': {k: 0 for k in k_values}, 'recall': {k: 0.0 for k in k_values}}
    
    for i in tqdm(range(0, total_queries, batch_process_size), desc="Batch Processing"):
        batch_end = min(i + batch_process_size, total_queries)
        batch_indices = range(i, batch_end)
        
        batch_pairs = []
        batch_meta = [] 
        
        for q_idx in batch_indices:
            q_text = queries[q_idx]
            gt_ids = ground_truths[q_idx]
            
            retrieved_indices = I[q_idx][:top_k_rerank]
            
            current_doc_ids = []
            current_doc_texts = []
            
            for doc_idx in retrieved_indices:
                if doc_idx != -1 and doc_idx < len(all_chunk_ids):
                    current_doc_ids.append(all_chunk_ids[doc_idx])
                    current_doc_texts.append(all_chunk_texts[doc_idx])
            
            if gt_ids:
                for k in k_values:
                    h, r = calculate_metrics(current_doc_ids, gt_ids, k)
                    metrics_retriever['hit'][k] += h
                    metrics_retriever['recall'][k] += r
            
            if current_doc_texts:
                for doc_text, doc_id in zip(current_doc_texts, current_doc_ids):
                    batch_pairs.append([q_text, doc_text])
                
                batch_meta.append({
                    'q_idx': q_idx,
                    'count': len(current_doc_texts),
                    'doc_ids': current_doc_ids
                })

        if batch_pairs:
            scores = cross_encoder.predict(batch_pairs, batch_size=128, show_progress_bar=False)
            
            cursor = 0
            for meta in batch_meta:
                q_idx = meta['q_idx']
                count = meta['count']
                doc_ids = meta['doc_ids']
                gt_ids = ground_truths[q_idx]
                
                q_scores = scores[cursor : cursor + count]
                cursor += count
                
                ranked_results = sorted(zip(q_scores, doc_ids), key=lambda x: x[0], reverse=True)
                reranked_ids = [d_id for _, d_id in ranked_results]
                
                if gt_ids:
                    for k in k_values:
                        h, r = calculate_metrics(reranked_ids, gt_ids, k)
                        metrics_reranker['hit'][k] += h
                        metrics_reranker['recall'][k] += r

    print("\n" + "="*30)
    print("FINAL RESULTS")
    print("="*30)
    
    def print_pretty(name, metrics):
        print(f"\n>> {name}:")
        print(f"{'K':<5} | {'Hit Rate':<10} | {'Recall':<10}")
        print("-" * 30)
        for k in k_values:
            hit = metrics['hit'][k] / total_queries
            recall = metrics['recall'][k] / total_queries
            print(f"{k:<5} | {hit:.4f}     | {recall:.4f}")

    print_pretty("Stage 1: Bi-Encoder (Retrieval)", metrics_retriever)
    print_pretty("Stage 2: Cross-Encoder (Reranking)", metrics_reranker)

In [ ]:
if __name__ == "__main__":
    BASE_INPUT_DIR = '/kaggle/input'
    TEST_DATA_PATH = os.path.join(BASE_INPUT_DIR, 'zalo-dataset/crawled_test_5k.jsonl')
    CORPUS_DATA_PATH = os.path.join(BASE_INPUT_DIR, 'zalo-dataset/corpus.csv')
    FAISS_INDEX_PATH = os.path.join(BASE_INPUT_DIR, 'qa-retrieval-rag-chunking/index.faiss')
    evaluate_retrieval_rag_final_optimized(test_data_path=TEST_DATA_PATH, corpus_data_path=CORPUS_DATA_PATH, faiss_index_path=FAISS_INDEX_PATH)

Using device: cuda

--- Loading Models ---


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Pre-processing corpus text...
Loading FAISS index...

--- Loading Test Data ---
Total Queries: 5000

--- Stage 1: Retrieval (Bi-Encoder) ---


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Searching Top 100 candidates...

--- Stage 2: Reranking & Eval (Processing 100 queries at a time) ---


Batch Processing:   0%|          | 0/50 [00:00<?, ?it/s]


FINAL RESULTS

>> Stage 1: Bi-Encoder (Retrieval):
K     | Hit Rate   | Recall    
------------------------------
1     | 0.3984     | 0.2428
5     | 0.6464     | 0.4760
10    | 0.7256     | 0.5679
30    | 0.8240     | 0.6940

>> Stage 2: Cross-Encoder (Reranking):
K     | Hit Rate   | Recall    
------------------------------
1     | 0.6116     | 0.3842
5     | 0.7886     | 0.5975
10    | 0.8094     | 0.6550
30    | 0.8240     | 0.6940
